In [27]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils import data
from torch import nn, optim
from random import random
from tqdm import tqdm
from cugdt.Tiff import *
from glob import glob
from torch.nn import Transformer
import torch.nn.functional as F

In [2]:
DATASET_PATH = r"H:\7.Eco_parameter\Hunan_LULC\Long_time_Landsat_dataset/"
COMBINE_PATH = r"H:\7.Eco_parameter\Hunan_LULC\LULC_result\Zone_temp\split_zone\Combine_Data"

TEMP_PATH = r"H:\7.Eco_parameter\Hunan_LULC\LULC_result\Zone_temp"
PROJECT_PATH = r"H:\7.Eco_parameter\Hunan_LULC\LULC_result\result"
INPUT_PATH = r"H:\7.Eco_parameter\Hunan_LULC\LULC_result\input_path"
MODEL_PATH = r"H:\7.Eco_parameter\Hunan_LULC\Best_model\best_model_UNet0.pth"  # 最佳的模型路径

im_data, GEO, PROJECT = ReadGeoTIFF(DATASET_PATH + 'concatenated_1990.tif')
WIDTH = im_data.shape[1]   # 研究区域影像的宽
HEIGHT = im_data.shape[2]   # 研究区域影像的高

SPLIT = 500   # 分块的大小

# X_max和Y_max分别是可以被SPLIT整除，并且大于影像长和宽的最小值
X_max = ((int(WIDTH / SPLIT)) + 1) * SPLIT
Y_max = ((int(HEIGHT / SPLIT)) + 1) * SPLIT

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [3]:
CHANGE_DATA_PATH = r"H:\7.Eco_parameter\Hunan_LULC\Sample\change.npy"
STABLE_DATA_PATH = r"H:\7.Eco_parameter\Hunan_LULC\Sample\balanced_stable.npy"
# 计算 平均值和标准差 ，便于归一化
stable_data = np.load(STABLE_DATA_PATH, allow_pickle=True)
change_data = np.load(CHANGE_DATA_PATH, allow_pickle=True)
data0 = np.concatenate((stable_data, change_data), axis=0)
# 提取要计算均值和标准差的波段
bands_data = data0[:, :6, :]
# 确保数据为数值类型
bands_data = bands_data.astype(np.float32)
bands_data = bands_data

# 计算每个波段的均值和标准差
mean = np.mean(bands_data, axis=(0, 2))
std = np.std(bands_data, axis=(0, 2))

print("均值:", mean)
print("标准差:", std)

均值: [ 914.1453 1094.7517 1133.948  2160.0928 1953.543  1468.8158]
标准差: [ 569.4535   656.3193   901.48566 1176.8358  1303.2179  1239.0968 ]


In [4]:
def CreatDir(path):
    # 创建空文件夹
    if not os.path.exists(path):
        os.makedirs(path)

def split_process(path, x_max, y_max, out_path, date):
    CreatDir(out_path)
    data, im_geo, im_pro = ReadGeoTIFF(path)
    for id_i, i in enumerate(range(0, x_max, SPLIT)):
        for id_j, j in enumerate(range(0, y_max, SPLIT)):
            x_start, x_end = i, i + SPLIT
            y_start, y_end = j, j + SPLIT
            im_data = data[:, x_start: x_end, y_start: y_end]   
            idx_value = np.where(im_data != 0)  # 有数据的区域
            if idx_value[0].shape[0] != 0:  # 如果不是空白区块
                # 每个切片都要修改仿射矩阵，左上角的坐标   
                im_geotrans = list(im_geo)
                im_geotrans[0] = im_geotrans[0] + int(y_start) * im_geotrans[1]
                im_geotrans[3] = im_geotrans[3] + int(x_start) * im_geotrans[5]

                CreatDir(os.path.join(out_path, f"zone_{id_i}_{id_j}"))
                CreateGeoTiff(os.path.join(out_path, f"zone_{id_i}_{id_j}", date),
                              im_data, tuple(im_geotrans), im_pro)

# print('Step1: 数据切片：')
# for i, file in enumerate(tqdm(os.listdir(DATASET_PATH))):   # 遍历所有时相的文件
#     if file.endswith('tif'):
#         filepath = os.path.join(DATASET_PATH, file)
#         split_process(filepath, X_max, Y_max, os.path.join(TEMP_PATH, 'split_zone', 'Data'), file)

In [5]:
out_path = os.path.join(TEMP_PATH, 'split_zone', 'Combine_Data')
CreatDir(out_path)
data_path = os.path.join(TEMP_PATH, 'split_zone', 'Data')
# print('Step2: 数据合并：')
# for zone in tqdm(os.listdir(data_path)):
#     for id_tif, tif in enumerate(os.listdir(os.path.join(data_path, zone))):
#         tif_path = os.path.join(data_path, zone, tif)
#         im_data, im_geo, im_pro = ReadGeoTIFF(tif_path)
#         if id_tif == 0:
#             res = im_data[None, :6, :, :]
#         else:
#             res = np.concatenate((res, im_data[None, :6, :, :]), axis=0)
#     t, b, x, y = res.shape
#     # CreateGeoTiff函数仅可以保存三维数据，所以暂时将时间维度和波段维度合并为一个维度
#     CreateGeoTiff(os.path.join(out_path, f'{zone}.tif'), res.reshape(t * b, x, y), im_geo, im_pro)    # 暂存到TEMP_PATH文件夹, 完整文件在COMBINE_PATH

In [6]:
class MaskDataset(data.Dataset):
    def __init__(self, paths):
        super(MaskDataset, self).__init__()
        self.image_paths = paths
        
    def __getitem__(self, index):
        data = self.image_paths[:, :, index].T
        # 推理的时候不再进行数据增强
        # 标准化        
        MEAN = mean
        STD = std
        return (data - MEAN[:, None]) / STD[:, None]

    def __len__(self):
        return self.image_paths.shape[2]

def load_data(path):
    im_data, im_geotrans, im_proj = ReadGeoTIFF(path)
    t, x, y = im_data.shape
    tif = im_data.reshape(-1, 6, x, y)  # 将数据恢复为四维
    tif = tif.reshape([tif.shape[0], tif.shape[1], -1])   # 将数据的空间维度压缩
    temp1 = np.zeros([tif.shape[0], tif.shape[2]])
    value_id = np.argwhere(tif[0, 0] != 0)   # 定位有数据的区域
    tif = np.squeeze(tif[:, :, value_id])
    BATCH_SIZE = 32
    test_ds = MaskDataset(paths=tif)
    test_dl = data.DataLoader(dataset=test_ds, batch_size=BATCH_SIZE)
    return test_dl, (x, y), temp1, value_id   # 数据集， 数据的长宽，空矩阵（用于存放结果），有效数据的位置（当数据空间分布不规则时间有用）

In [7]:
class UNet(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet, self).__init__()
        n=6
        self.encoder1 = self.conv_block(in_channels, 2**n)
        self.encoder2 = self.conv_block(2**n, 2**(n+1))
        self.encoder3 = self.conv_block(2**(n+1), 2**(n+2))
        self.encoder4 = self.conv_block(2**(n+2), 2**(n+3))
        self.bottleneck = self.conv_block(2**(n+3), 2**(n+4))
        self.upconv4 = nn.ConvTranspose1d(2**(n+4), 2**(n+3), kernel_size=2, stride=2)
        self.decoder4 = self.conv_block(2**(n+4), 2**(n+3))
        self.upconv3 = nn.ConvTranspose1d(2**(n+3), 2**(n+2), kernel_size=2, stride=2)
        self.decoder3 = self.conv_block(2**(n+3), 2**(n+2))
        self.upconv2 = nn.ConvTranspose1d(2**(n+2), 2**(n+1), kernel_size=2, stride=2)
        self.decoder2 = self.conv_block(2**(n+2), 2**(n+1))
        self.upconv1 = nn.ConvTranspose1d(2**(n+1), 2**n, kernel_size=2, stride=2)
        self.decoder1 = self.conv_block(2**(n+1), 2**n)
        self.final_conv = nn.Conv1d(2**n, out_channels, kernel_size=1)

    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(F.max_pool1d(enc1, 2))
        enc3 = self.encoder3(F.max_pool1d(enc2, 2))
        enc4 = self.encoder4(F.max_pool1d(enc3, 2))
        bottleneck = self.bottleneck(F.max_pool1d(enc4, 2))
        
        dec4 = self.upconv4(bottleneck)
        dec4 = F.interpolate(dec4, size=enc4.size()[2:], mode='nearest')  # 调整尺寸
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.decoder4(dec4)
        
        dec3 = self.upconv3(dec4)
        dec3 = F.interpolate(dec3, size=enc3.size()[2:], mode='nearest')  # 调整尺寸
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3(dec3)
        
        dec2 = self.upconv2(dec3)
        dec2 = F.interpolate(dec2, size=enc2.size()[2:], mode='nearest')  # 调整尺寸
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)
        
        dec1 = self.upconv1(dec2)
        dec1 = F.interpolate(dec1, size=enc1.size()[2:], mode='nearest')  # 调整尺寸
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)
        
        return self.final_conv(dec1)

In [18]:
# 这里推理剩余部分没推理的
def format_num(n):
    # 格式化数字
    if n < 10:
        return '0' + str(n)
    else:
        return str(n)


# 推理真实的tif，分块合并的结果已保存在COMBINE_PATH
for idx, tif in enumerate(os.listdir(r"H:\7.Eco_parameter\Hunan_LULC\LULC_result\Zone_temp\split_zone\other")):
    if tif.endswith('tif'):
        out_path = os.path.join(INPUT_PATH, 'Result', tif)
        CreatDir(out_path)
        test_dl, tif_shape, res_temp, value_id = load_data(os.path.join(COMBINE_PATH, tif))   # 加载数据集
        model = UNet(6, 6).to(DEVICE)   # 定义模型
        model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))    # 加载训练好的模型参数
        with torch.no_grad():
            test_tqdm = tqdm(iterable=test_dl, total=len(test_dl))
            index = 0
            name = 0
            c = 0  # 记录轮次
            for test_images in test_tqdm:
                # print(test_images)
                test_pred = model(test_images.to(DEVICE).float())
                
                # 这里是一个推理的技巧，将序列调转方向输入，将两个结果取合
                test_pred_flip = torch.flip(model(torch.flip(test_images, dims=[2]).to(DEVICE).float()), dims=[2])
                test_pred += test_pred_flip

                pred = torch.argmax(input=test_pred, dim=1)
                if index == 0:
                    res = pred.cpu().numpy()
                else:
                    res = np.concatenate((res, pred.cpu().numpy()))
                index += 1
                if index == 500:   # 这里使用了一个技巧，如果一次性写入48*500*500的数据的话会比较慢，所以一般可以分阶段保存结果，这里就是每计算500*128个像素那么则保存一次结果
                    np.save(os.path.join(out_path, 'res_{}.npy'.format(format_num(name))), res)
                    name += 1
                    index = 0
                if c == len(test_tqdm) - 1 and index != 0:
                    np.save(os.path.join(out_path, 'res_{}.npy'.format(format_num(name))), res)
                c += 1

        # 将多个结果合并
        _temp = np.array([0])
        i = 0
        for filepath in glob(os.path.join(out_path, 'res_*.npy')):
            res_data = np.load(filepath)
            print(f"Loaded {filepath} with shape {res_data.shape}")
            if i == 0:
                _temp = res_data
            else:
                _temp = np.concatenate((_temp, res_data))
            i += 1


        res_temp = res_temp.T
        if _temp.shape[0] != 0:
            res_temp[value_id[:, 0]] = _temp
            print(f"tif_shape: {tif_shape}, _temp shape: {_temp.shape}")

        result = res_temp.reshape([tif_shape[0], tif_shape[1], 34]).T
        np.save(os.path.join(out_path, 'result.npy'), result)   # 结果暂存至INPUT_PATH文件夹，完整文件在DATASET_PATH + 'Result'

100%|██████████| 204/204 [00:01<00:00, 161.37it/s]


Loaded H:\7.Eco_parameter\Hunan_LULC\LULC_result\input_path\Result\zone_37_27.tif\res_00.npy with shape (6507, 34)
tif_shape: (500, 500), _temp shape: (6507, 34)


In [ ]:
# def format_num(n):
#     # 格式化数字
#     if n < 10:
#         return '0' + str(n)
#     else:
#         return str(n)


# # 推理真实的tif，分块合并的结果已保存在COMBINE_PATH
# for idx, tif in enumerate(os.listdir(COMBINE_PATH)):
#     if tif.endswith('tif'):
#         out_path = os.path.join(INPUT_PATH, 'Result', tif)
#         CreatDir(out_path)
#         test_dl, tif_shape, res_temp, value_id = load_data(os.path.join(COMBINE_PATH, tif))   # 加载数据集
#         model = UNet(6, 6).to(DEVICE)   # 定义模型
#         model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))    # 加载训练好的模型参数
#         with torch.no_grad():
#             test_tqdm = tqdm(iterable=test_dl, total=len(test_dl))
#             index = 0
#             name = 0
#             c = 0  # 记录轮次
#             for test_images in test_tqdm:
#                 # print(test_images)
#                 test_pred = model(test_images.to(DEVICE).float())
                
#                 # 这里是一个推理的技巧，将序列调转方向输入，将两个结果取合
#                 test_pred_flip = torch.flip(model(torch.flip(test_images, dims=[2]).to(DEVICE).float()), dims=[2])
#                 test_pred += test_pred_flip

#                 pred = torch.argmax(input=test_pred, dim=1)
#                 if index == 0:
#                     res = pred.cpu().numpy()
#                 else:
#                     res = np.concatenate((res, pred.cpu().numpy()))
#                 index += 1
#                 if index == 500:   # 这里使用了一个技巧，如果一次性写入48*500*500的数据的话会比较慢，所以一般可以分阶段保存结果，这里就是每计算500*128个像素那么则保存一次结果
#                     np.save(os.path.join(out_path, 'res_{}.npy'.format(format_num(name))), res)
#                     name += 1
#                     index = 0
#                 if c == len(test_tqdm) - 1 and index != 0:
#                     np.save(os.path.join(out_path, 'res_{}.npy'.format(format_num(name))), res)
#                 c += 1

#         # 将多个结果合并
#         _temp = np.array([0])
#         i = 0
#         for filepath in glob(os.path.join(out_path, 'res_*.npy')):
#             res_data = np.load(filepath)
#             if i == 0:
#                 _temp = res_data
#             else:
#                 _temp = np.concatenate((_temp, res_data))
#             i += 1

#         res_temp = res_temp.T
#         if _temp.shape[0] != 0:
#             res_temp[value_id[:, 0]] = _temp
#         result = res_temp.reshape([tif_shape[0], tif_shape[1], 34]).T
#         np.save(os.path.join(out_path, 'result.npy'), result)   # 结果暂存至INPUT_PATH文件夹，完整文件在DATASET_PATH + 'Result'

In [28]:
# ... existing code ...
X_max = ((int(WIDTH / SPLIT)) + 1) * SPLIT
Y_max = ((int(HEIGHT / SPLIT)) + 1) * SPLIT
x_split = [i for i in range(0, X_max + 1, SPLIT)]
y_split = [i for i in range(0, Y_max + 1, SPLIT)]
RESULT_TEMP_PATH = r"H:\7.Eco_parameter\Hunan_LULC\LULC_result\input_path\Result"   # 已计算完的结果

# 每5年保存一次
years_per_file = 5
num_files = 34 // years_per_file

for file_idx in range(num_files):
    start_year = file_idx * years_per_file
    end_year = start_year + years_per_file
    temp = np.empty([years_per_file, WIDTH, HEIGHT], dtype=np.uint8)
    
    for i in tqdm(os.listdir(RESULT_TEMP_PATH)):
        data = np.load(os.path.join(RESULT_TEMP_PATH, i, 'result.npy'))
        zonex, zoney = int(i[:-4].split('_')[1]), int(i[:-4].split('_')[2])  # 获取zone编号
        xs, xe = x_split[zonex], x_split[zonex + 1]
        ys, ye = y_split[zoney], y_split[zoney + 1]
        temp[:, xs:xe, ys:ye] = data[start_year:end_year].transpose(0, 2, 1)
    
    CreateGeoTiff(os.path.join(PROJECT_PATH, f'monthly_LULC_result_{start_year}.tif'), 
                  temp, GEO, PROJECT, 'int', True)  # 压缩保存，导出至PROJECT_PATH

100%|██████████| 1190/1190 [18:11<00:00,  1.09it/s]


In [29]:
# ... existing code ...
X_max = ((int(WIDTH / SPLIT)) + 1) * SPLIT
Y_max = ((int(HEIGHT / SPLIT)) + 1) * SPLIT
x_split = [i for i in range(0, X_max + 1, SPLIT)]
y_split = [i for i in range(0, Y_max + 1, SPLIT)]
RESULT_TEMP_PATH = r"H:\7.Eco_parameter\Hunan_LULC\LULC_result\input_path\Result"   # 已计算完的结果

# 每5年保存一次
years_per_file = 5
num_files = 34 // 5

# 保存最后4年的数据
last_years = 4
start_year = 34 - last_years  # 计算最后4年的起始年份
end_year = 34  # 结束年份为34

# 创建一个临时数组来存放最后4年的数据
temp = np.empty([last_years, WIDTH, HEIGHT], dtype=np.uint8)

for i in tqdm(os.listdir(RESULT_TEMP_PATH)):
    data = np.load(os.path.join(RESULT_TEMP_PATH, i, 'result.npy'))
    zonex, zoney = int(i[:-4].split('_')[1]), int(i[:-4].split('_')[2])  # 获取zone编号
    xs, xe = x_split[zonex], x_split[zonex + 1]
    ys, ye = y_split[zoney], y_split[zoney + 1]
    
    # 提取最后4年的数据并存储
    temp[:, xs:xe, ys:ye] = data[start_year:end_year].transpose(0, 2, 1)

# 保存最后4年的数据为 GeoTIFF
CreateGeoTiff(os.path.join(PROJECT_PATH, f'monthly_LULC_result_{start_year}.tif'), 
              temp, GEO, PROJECT, 'int', True)  # 压缩保存，导出至PROJECT_PATH

100%|██████████| 1190/1190 [18:03<00:00,  1.10it/s]


In [31]:
import numpy as np
import rasterio
import os

input_file = r"H:\7.Eco_parameter\Hunan_LULC\LULC_result\result\HN_LC_result.tif"

# 打开遥感影像
with rasterio.open(input_file) as src:
    # 读取所有波段
    data = src.read()  # shape: (bands, height, width)

# 确保数据有 34 个波段
if data.shape[0] != 34:
    raise ValueError("影像波段数量不为 34")

# 定义年份范围
years = range(1990, 2024)  # 1990 到 2023

# 输出目录
output_dir = r"H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC"
os.makedirs(output_dir, exist_ok=True)  # 确保输出目录存在

# 保存每年的数据
for i, year in enumerate(years):
    # 提取第 i 个波段（i 从 0 开始，所以对应年份为 1990 + i）
    year_data = data[i]  # shape: (height, width)

    # 定义输出文件名
    output_file = os.path.join(output_dir, f"LC_{year}.tif")

    # 保存每年的数据
    with rasterio.open(
        output_file,
        'w',
        driver='GTiff',
        height=year_data.shape[0],
        width=year_data.shape[1],
        count=1,
        dtype=year_data.dtype,
        crs=src.crs,
        transform=src.transform
    ) as dst:
        dst.write(year_data, 1)  # 写入第一个波段

    print(f"保存 {year} 的数据到 {output_file}")

保存 1990 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_1990.tif
保存 1991 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_1991.tif
保存 1992 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_1992.tif
保存 1993 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_1993.tif
保存 1994 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_1994.tif
保存 1995 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_1995.tif
保存 1996 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_1996.tif
保存 1997 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_1997.tif
保存 1998 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_1998.tif
保存 1999 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_1999.tif
保存 2000 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_2000.tif
保存 2001 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_2001.tif
保存 2002 的数据到 H:\7.Eco_parameter\Hunan_LULC\LULC_result\annual_LC\LC_2002.tif

In [ ]:
# import os
# import shutil

# # 定义文件夹路径
# result_folder = r'H:\7.Eco_parameter\Hunan_LULC\LULC_result\input_path\Result'
# combine_data_folder = r'H:\7.Eco_parameter\Hunan_LULC\LULC_result\Zone_temp\split_zone\Combine_Data'
# target_folder = r'H:\7.Eco_parameter\Hunan_LULC\LULC_result\Zone_temp\split_zone\other'

# # 获取两个文件夹中的文件名（不含路径）
# files_in_result = set(os.listdir(result_folder))
# files_in_combine = set(os.listdir(combine_data_folder))

# # 找出只存在于 Combine_Data 中而不存在于 Result 文件夹中的文件
# files_to_copy = files_in_combine - files_in_result

# # 确保目标文件夹存在
# os.makedirs(target_folder, exist_ok=True)

# # 复制文件到目标文件夹
# for file_name in files_to_copy:
#     src = os.path.join(combine_data_folder, file_name)
#     dst = os.path.join(target_folder, file_name)

#     # 检查文件是否存在，避免错误
#     if os.path.exists(src):
#         shutil.copy(src, dst)
#         print(f"Copied: {file_name} to {target_folder}")
#     else:
#         print(f"File not found: {file_name}")

# print("All missing files have been copied.")